In [1]:
pip install fredapi

Note: you may need to restart the kernel to use updated packages.


In [6]:
from fredapi import Fred
import pandas as pd

# ===========================================
# SUA API KEY
# ===========================================

fred = Fred(api_key="7fe3354b4adf9d1af3683498ae46688f")

# ===========================================
# Séries
# ===========================================

series = {
    # Mercado Financeiro
    "vix": "VIXCLS",                     # CBOE VIX
    "sp500": "SP500",                    # S&P500
    "gs10": "GS10",                      # Treasury 10Y
    "tb3ms": "TB3MS",                    # Treasury 3M
    "fedfunds": "FEDFUNDS",              # Federal Funds Rate

    # Economia
    "industrial_production": "INDPRO",   # Industrial Production
    "cpi": "CPIAUCSL",                   # CPI
    "unemployment": "UNRATE",            # Unemployment Rate

    # Condições Financeiras
    "nfci": "NFCI",                      # Chicago Fed NFCI
    "epu": "USEPUINDXM"                  # Economic Policy Uncertainty
}

# ===========================================
# Download
# ===========================================

data = {}

for name, code in series.items():
    print(f"Baixando {name}")
    data[name] = fred.get_series(
        code,
        observation_start="2010-01-01",
        observation_end="2025-12-31"
    )

df_macro = pd.DataFrame(data)

# ===========================================
# Datas
# ===========================================

df_macro.index = pd.to_datetime(df_macro.index)

# ===========================================
# Transformar para frequência mensal
# ===========================================

df_macro = df_macro.resample("ME").last()

# ===========================================
# Variáveis derivadas
# ===========================================

# Inclinação da curva
df_macro["term_spread"] = (
    df_macro["gs10"] -
    df_macro["tb3ms"]
)


# Retorno mensal do S&P500
df_macro["sp500_return"] = (
    df_macro["sp500"]
    .pct_change()
)

# Crescimento mensal da produção industrial
df_macro["industrial_growth"] = (
    df_macro["industrial_production"]
    .pct_change()
)

# Inflação mensal
df_macro["inflation_monthly"] = (
    df_macro["cpi"]
    .pct_change()
)

# ===========================================
# Resetar índice
# ===========================================

df_macro = df_macro.reset_index()

df_macro.rename(columns={"index": "date"}, inplace=True)

print(df_macro.head(10))

Baixando vix
Baixando sp500
Baixando gs10
Baixando tb3ms
Baixando fedfunds
Baixando industrial_production
Baixando cpi
Baixando unemployment
Baixando nfci
Baixando epu
        date    vix  sp500  gs10  tb3ms  fedfunds  industrial_production  \
0 2010-01-31  24.62    NaN  3.73   0.06      0.11                89.3426   
1 2010-02-28  19.50    NaN  3.69   0.11      0.13                89.6779   
2 2010-03-31  17.59    NaN  3.73   0.15      0.16                90.2928   
3 2010-04-30  22.05    NaN  3.85   0.16      0.20                90.5991   
4 2010-05-31  32.07    NaN  3.42   0.16      0.20                91.8230   
5 2010-06-30  34.54    NaN  3.20   0.12      0.18                91.9928   
6 2010-07-31  23.50    NaN  3.01   0.16      0.18                92.3421   
7 2010-08-31  26.05    NaN  2.70   0.16      0.19                92.6805   
8 2010-09-30  23.70    NaN  2.65   0.15      0.19                92.9537   
9 2010-10-31  21.20    NaN  2.54   0.13      0.19                92.6903

In [7]:
missing = pd.DataFrame({
    "missing": df_macro.isna().sum(),
    "percent": (df_macro.isna().mean() * 100).round(2)
})

missing = missing.sort_values("percent", ascending=False)

print(missing)

                       missing  percent
sp500_return                78    40.62
sp500                       77    40.10
inflation_monthly            3     1.56
unemployment                 1     0.52
cpi                          1     0.52
industrial_growth            1     0.52
date                         0     0.00
industrial_production        0     0.00
fedfunds                     0     0.00
tb3ms                        0     0.00
gs10                         0     0.00
vix                          0     0.00
nfci                         0     0.00
epu                          0     0.00
term_spread                  0     0.00


In [9]:
sp500 = fred.get_series(
    "SP500",
    observation_start="2010-01-01",
    observation_end="2025-12-31"
)

sp500 = (
    sp500
    .ffill()
    .resample("ME")
    .last()
)

sp500_return = sp500.pct_change()

In [12]:
print(df_macro["sp500_return"].isna().sum())

78


In [13]:
df_macro.loc[df_macro["sp500_return"].isna(), ["date", "sp500", "sp500_return"]]

,date,sp500,sp500_return
0,2010-01-31,NaN,NaN
1,2010-02-28,NaN,NaN
2,2010-03-31,NaN,NaN
3,2010-04-30,NaN,NaN
4,2010-05-31,NaN,NaN
...,...,...,...
73,2016-02-29,NaN,NaN
74,2016-03-31,NaN,NaN
75,2016-04-30,NaN,NaN
76,2016-05-31,NaN,NaN
